# `biophys_interop` — a 5-minute tour

**Make messy biophysical measurements computable.** This notebook runs the toolkit end-to-end on a few real records so
you can see, without installing anything locally, exactly what it does:

1. **`standardize()`** — put a raw record into one schema, with SI units.
2. **`qc()`** — run 72 transparent, method-aware rules and attach a `pass` / `warn` / `fail` flag *with reasons*.
3. **`featurize()`** — produce a fixed-width vector (plus a mask marking untrusted fields) for a downstream model.

There is no trained model anywhere in the quality-control step — the checks are simple, auditable rules a scientist can
read and argue with.

> **Running in Google Colab?** Just press *Runtime → Run all*. The first cell installs the package.


In [1]:
# Install the current public toolkit directly from GitHub. After a PyPI release, use:
#   pip install "biophys_interop[batch]"
# The [batch] extra adds pandas + pyarrow for the CSV/TSV workflow (Parquet output).
# Use check=True: a failed installation must stop the walkthrough rather than reuse an unknown environment.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "biophys_interop[batch] @ git+https://github.com/xingaobio/biophys_interop.git@master"], check=True)

import biophys_interop as bi
print("biophys_interop version:", bi.__version__)
print("modalities supported:", len(bi.MODALITIES), "| feature vector dimension:", bi.FEATURE_DIM)

biophys_interop version: 0.1.0
modalities supported: 24 | feature vector dimension: 64


## 1 · One record, three steps

We start with a single surface-plasmon-resonance (SPR) measurement, written the way it might come out of an instrument
or a spreadsheet: a dissociation constant `K_D` of 2 nM, a tested ("trace") concentration of 5 nM, at 25 °C.

Watch what each step does to it.

In [2]:
from biophys_interop import standardize, qc, featurize

raw = {
    "record_id": "r1", "modality": "SPR", "entity_id": "P00519",
    "assay_type": "SPR", "temperature_C": 25,
    "measurement": {"KD": {"value": 2.0, "unit": "nM"},
                    "trace_conc": {"value": 5.0, "unit": "nM"}},
}

# Step 1 — standardize: canonical schema + SI units
rec = standardize(raw, "SPR")
print("standardized measurement:", rec["measurement"])
print("standardized conditions :", rec.get("conditions"))

standardized measurement: {'KD': 2e-09, 'trace_conc': 5e-09}
standardized conditions : {'temperature_K': 298.15, 'pH': None, 'buffer': None}


`K_D` is now `2e-09` M and the temperature is `298.15` K — everything is in SI units, so records from different
instruments become directly comparable.

Now the quality check:

In [3]:
# Step 2 — qc: run the rule registry, attach a flag + human-readable reasons
rec = qc(rec)
print("QC flag :", rec["qc"]["flag"])
print("reasons :")
for r in rec["qc"]["reasons"]:
    print(f"   - {r['code']:<34} ({r.get('severity')})")

QC flag : fail
reasons :
   - titration_regime                   (fail)
   - steady_state_no_equilibration_proof (warn)
   - active_fraction_unknown            (warn)
   - ph_unreported                      (warn)
   - buffer_unreported                  (warn)
   - uncertainty_unreported             (warn)
   - replicate_unreported               (warn)
   - entity_sequence_missing            (warn)
   - provenance_incomplete              (warn)
   - method_assay_mismatch              (warn)


The record is flagged **`fail`**, and the layer says *why*: the top reason is `titration_regime` — the tested
concentration (5 nM) is **not** far below the reported `K_D` (2 nM), so this experiment physically cannot resolve
binding that tight. That is exactly the kind of silent problem that pollutes pooled affinity datasets. The remaining
reasons note missing metadata (pH, buffer, active fraction) that would be needed to trust the number.

Finally, turn it into something a model can consume:

In [4]:
# Step 3 — featurize: fixed-width vector + a mask of which fields are trustworthy
fx = featurize(rec)
print("feature vector length:", len(fx["vector"]))
print("populated features   :", sum(1 for v in fx["vector"] if v))
print("masked (missing) dims :", sum(1 for m in fx["mask"] if not m))

feature vector length: 64
populated features   : 7
masked (missing) dims : 56


The vector is always the same width (64), regardless of technique, so records from SPR, ITC, SAXS, and the rest all
live in one feature space. The **mask** tells the model which of those 64 slots are real for this record and which are
missing — so absent metadata is never silently treated as a measured zero.

## 2 · The checks are method-aware: an ITC example

The rules know what each technique should look like. Here is an isothermal-titration-calorimetry (ITC) record. Note the
input contract: for ITC, `K_D` is given as `{value, unit}`, while the thermodynamic terms `dH`, `dS`, `n` are plain
numbers.

In [5]:
itc = {
    "record_id": "r2", "modality": "ITC", "entity_id": "P00519",
    "assay_type": "ITC", "temperature_C": 25, "pH": 7.4,
    "buffer": "PBS 20 mM, 150 mM NaCl",
    "measurement": {"KD": {"value": 50.0, "unit": "nM"},
                    "dH": -8.0, "dS": -4.2, "n": 1.0},
}

rec_itc = qc(standardize(itc, "ITC"))
print("ITC flag :", rec_itc["qc"]["flag"])
for r in rec_itc["qc"]["reasons"]:
    print(f"   - {r['code']:<28} ({r.get('severity')})")

ITC flag : warn
   - itc_dG_KD_mismatch           (warn)
   - uncertainty_unreported       (warn)
   - replicate_unreported         (warn)
   - entity_sequence_missing      (warn)
   - provenance_incomplete        (warn)
   - method_assay_mismatch        (warn)


The first reason, `itc_dG_KD_mismatch`, is a **thermodynamic-consistency check**: the reported enthalpy/entropy imply a
binding free energy that does not match the reported `K_D`. This is a genuine internal contradiction that a generic
range check would miss — and it is the kind of rule that only makes sense once you know the record came from ITC.

(These method-specific biophysical rules are literature-grounded but, unlike the two general affinity rules in the
paper, have not yet been validated against a curated benchmark — see the manuscript's scope note.)

## 3 · A whole dataset at once: the `batch` command

For real work you do not process records one at a time — you point the toolkit at a CSV or TSV. The `batch` command
runs every row through `standardize → qc` and writes three things: a canonical table, a per-record QC report, and a
manifest that records the input hash and the tool version (so a cleaned dataset can be regenerated and audited later).

Here we run it on a 200-row sample of real ChEMBL records shipped with the package.

In [6]:
# Equivalent shell command:  biophys_interop batch demo_input.csv --outdir out/
from biophys_interop.cli import main
import pathlib

# (Colab) grab the bundled demo CSV from the repo if it isn't alongside the notebook.
DEMO_URL = "https://raw.githubusercontent.com/xingaobio/biophys_interop/master/examples/demo_input.csv"
csv_path = "demo_input.csv"
if not pathlib.Path(csv_path).exists():
    try:
        import urllib.request
        urllib.request.urlretrieve(DEMO_URL, csv_path)
    except Exception as e:
        print("Provide a demo_input.csv (record_id,modality,target,molecule,type,value_nM,relation,pchembl).", e)

main(["batch", csv_path, "--modality", "other", "--outdir", "out"])

[batch] 200 records -> out/
  canonical : canonical.parquet (parquet)
  qc_report : qc_report.json  (flags: {'warn': 200})
  manifest  : manifest.json   (input sha256 c0ce4c5057e5bc04…)


In [7]:
# Inspect what batch produced
import json
report = json.load(open("out/qc_report.json"))
manifest = json.load(open("out/manifest.json"))
print("records processed :", report["n_records"])
print("QC flag counts    :", report["flag_counts"])
print("tool + version    :", manifest["tool"], manifest["version"])
print("input sha256      :", manifest["input_sha256"][:16], "…")

records processed : 200
QC flag counts    : {'warn': 200}
tool + version    : biophys_interop 0.1.0
input sha256      : c0ce4c5057e5bc04 …


Every one of the 200 minimal ChEMBL rows comes back `warn` — correctly, because they carry only a value and no
experimental metadata (temperature, pH, buffer, replicate count), so the completeness rules all fire. That is the
honest verdict: the *number* may be fine, but without conditions you cannot fully trust or compare it.

> **Note on output format:** with the `[batch]` extra installed you get `canonical.parquet`; without `pyarrow` the
> toolkit falls back to `canonical.jsonl`. Both contain the same records.

## Where to go next

- **Paper / preprint:** [the v1 ChemRxiv preprint](https://doi.org/10.26434/chemrxiv.15006416/v1) describes the
  method, with an important boundary: ChEMBL annotations are benchmark proxies, not independently verified source
  errors. The repository README records the current evidence and limitations.
- **`batch` on your own data:** any CSV/TSV with a `value_nM` column works; add `temperature_C`, `pH`, `buffer`,
  replicate counts and the completeness rules stop firing.
- **Use the QC flag downstream:** filter records, weight them in training, or simply audit a dataset before use.

The quality-control step is deterministic and model-free — you can read every rule, and every flag comes with its
reason.